# Health Data Quality Assessment - Walkthrough

This notebook demonstrates the complete Data Quality Assessment (DQA) pipeline for routine health facility reports in Nigeria.

## Contents
1. Setup and Data Generation
2. Load and Explore Data
3. Quality Checks (Step-by-Step)
4. Scoring and Metrics
5. Visualizations
6. Summary and Recommendations

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from pathlib import Path
import sys

# Add src to path
sys.path.insert(0, '../')

from src.quality import rules, metrics
from src.utils import io, dates

# Visualization settings
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Create figures directory
Path('../reports/figures').mkdir(parents=True, exist_ok=True)

print("✓ Setup complete")

## 1. Setup and Data Generation

First, let's generate synthetic data if it doesn't exist yet.

In [ ]:
# Check if data exists, generate if not
reports_path = Path('../data/raw/facility_reports.csv')

if not reports_path.exists():
    print("Generating synthetic data...")
    !python -m src.data.generate_synthetic_routine_data --facilities 1200 --months 24
else:
    print("✓ Data already exists")

## 2. Load and Explore Data

In [ ]:
# Load data
print("Loading data...")
registry = pd.read_csv('../data/raw/facility_registry.csv')
reports = pd.read_csv('../data/raw/facility_reports.csv')
config = yaml.safe_load(open('../config/dqa_config.yml', 'r'))

print(f"\n✓ Loaded:")
print(f"  - {len(registry)} facilities")
print(f"  - {len(reports)} facility-month reports")
print(f"  - {reports['period'].nunique()} unique periods")

In [ ]:
# Display sample records
print("Sample Facility Registry:")
display(registry.head())

print("\nSample Facility Reports:")
display(reports.head())

In [ ]:
# Basic statistics
print("Facility Distribution by State:")
state_counts = registry['state'].value_counts().sort_index()
print(state_counts)

print(f"\nFacility Types:")
print(registry['facility_type'].value_counts())

print(f"\nOwnership:")
print(registry['ownership'].value_counts())

## 3. Quality Checks (Step-by-Step)

Now let's run each quality check and examine the results.

### 3.1 Completeness Check

In [ ]:
# Merge reports with registry for state information
reports_merged = reports.merge(registry[['facility_id', 'state', 'facility_type']], 
                               on='facility_id', how='left')

# Check completeness
completeness_results = rules.check_completeness(
    reports_merged, 
    config['essential_columns'],
    threshold=config['thresholds']['completeness_min']
)

print("Completeness Check Results:")
print(f"  - Total records checked: {len(completeness_results)}")
print(f"  - Records flagged incomplete: {completeness_results['flag_incomplete'].sum()}")
print(f"  - Average completeness rate: {completeness_results['completeness_rate'].mean():.2%}")

print("\nSample incomplete records:")
display(completeness_results[completeness_results['flag_incomplete'] == True].head())

### 3.2 Duplicate Detection

In [ ]:
# Check for duplicates
duplicates_results = rules.check_duplicates(reports_merged)

print("Duplicate Detection Results:")
print(f"  - Total facility-months: {len(duplicates_results)}")
print(f"  - Facility-months with duplicates: {duplicates_results['flag_duplicate'].sum()}")
print(f"  - Duplicate rate: {duplicates_results['flag_duplicate'].sum() / len(duplicates_results):.2%}")

print("\nSample duplicate records:")
display(duplicates_results[duplicates_results['flag_duplicate'] == True].head())

### 3.3 Timeliness Check

In [ ]:
# Check timeliness
timeliness_results = rules.check_timeliness(
    reports_merged,
    due_days=config['thresholds']['timeliness_due_days']
)

print("Timeliness Check Results:")
print(f"  - Total records: {len(timeliness_results)}")
print(f"  - Records submitted late: {timeliness_results['flag_late'].sum()}")
print(f"  - Late submission rate: {timeliness_results['flag_late'].sum() / len(timeliness_results):.2%}")
print(f"  - Average days late (for late records): {timeliness_results[timeliness_results['flag_late']]['days_late'].mean():.1f}")

print("\nDistribution of days late:")
print(timeliness_results['days_late'].describe())

### 3.4 Outlier Detection

In [ ]:
# Get indicator columns
indicator_cols = [col for col in config['essential_columns'] 
                 if col not in ['facility_id', 'period', 'submission_date']]

# Check for outliers
outliers_results = rules.check_outliers(
    reports_merged,
    indicator_cols,
    threshold=config['thresholds']['outliers_mad_threshold']
)

print("Outlier Detection Results:")
print(f"  - Total indicator-observations checked: {len(outliers_results)}")
print(f"  - Outliers detected: {outliers_results['flag_outlier'].sum()}")
print(f"  - Outlier rate: {outliers_results['flag_outlier'].sum() / len(outliers_results):.2%}")

print("\nOutliers by indicator:")
outlier_by_indicator = outliers_results[outliers_results['flag_outlier']].groupby('indicator').size()
print(outlier_by_indicator.sort_values(ascending=False))

print("\nSample outlier records:")
display(outliers_results[outliers_results['flag_outlier'] == True].head())

### 3.5 Spike Detection

In [ ]:
# Check for spikes
spikes_results = rules.check_spikes(
    reports_merged,
    indicator_cols,
    pct_change_hi=config['thresholds']['spikes_pct_change_hi'],
    pct_change_lo=config['thresholds']['spikes_pct_change_lo']
)

print("Spike Detection Results:")
print(f"  - Total month-over-month comparisons: {len(spikes_results)}")
print(f"  - Spikes detected: {spikes_results['flag_spike'].sum()}")
print(f"  - Spike rate: {spikes_results['flag_spike'].sum() / len(spikes_results):.2%}")

print("\nSpikes by indicator:")
spike_by_indicator = spikes_results[spikes_results['flag_spike']].groupby('indicator').size()
print(spike_by_indicator.sort_values(ascending=False))

print("\nSample spike records:")
display(spikes_results[spikes_results['flag_spike'] == True].head())

### 3.6 Consistency Check

In [ ]:
# Check consistency
consistency_results = rules.check_consistency(reports_merged)

print("Consistency Check Results:")
print(f"  - Total records checked: {len(consistency_results)}")
print(f"  - Records with violations: {consistency_results['flag_inconsistent'].sum()}")
print(f"  - Violation rate: {consistency_results['flag_inconsistent'].sum() / len(consistency_results):.2%}")

print("\nViolation counts:")
print(consistency_results['violation_count'].value_counts().sort_index())

print("\nSample violation records:")
display(consistency_results[consistency_results['flag_inconsistent'] == True].head())

## 4. Scoring and Metrics

Now let's compute quality scores and create summaries.

In [ ]:
# Collect all check results
check_results = {
    'completeness': completeness_results,
    'duplicates': duplicates_results,
    'timeliness': timeliness_results,
    'outliers': outliers_results,
    'spikes': spikes_results,
    'consistency': consistency_results
}

# Compute metrics
output_dir = Path('../data/processed')
summaries = metrics.compute_all_metrics(check_results, registry, config, output_dir)

print("✓ Metrics computed and saved")

In [ ]:
# Display detailed results sample
detailed = summaries['detailed']
print("Detailed Results (facility-month level):")
display(detailed.head())

print("\nSummary Statistics:")
score_cols = [col for col in detailed.columns if col.endswith('_score')]
print(detailed[score_cols].describe())

In [ ]:
# State-level summary
state_summary = summaries['state']
print("State-Level Summary:")
display(state_summary.sort_values('overall_score', ascending=False).head(10))

## 5. Visualizations

Let's create visualizations to understand data quality patterns.

In [ ]:
# Overall score distribution
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(detailed['overall_score'], bins=30, edgecolor='black', alpha=0.7)
ax.axvline(detailed['overall_score'].mean(), color='red', linestyle='--', 
           label=f"Mean: {detailed['overall_score'].mean():.1f}")
ax.set_xlabel('Overall Quality Score')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Overall Quality Scores')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/overall_score_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: overall_score_distribution.png")

In [ ]:
# Quality score components
score_cols = [col for col in detailed.columns if col.endswith('_score')]
score_means = detailed[score_cols].mean().sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
score_means.plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Average Score (0-100)')
ax.set_ylabel('Quality Component')
ax.set_title('Average Quality Scores by Component')
ax.set_xlim(0, 100)
ax.axvline(75, color='red', linestyle='--', alpha=0.5, label='Target: 75')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/quality_components.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: quality_components.png")

In [ ]:
# Top and bottom states
top_10 = state_summary.nlargest(10, 'overall_score')[['state', 'overall_score']]
bottom_10 = state_summary.nsmallest(10, 'overall_score')[['state', 'overall_score']]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Top 10
top_10.plot(kind='barh', x='state', y='overall_score', ax=ax1, color='green', legend=False)
ax1.set_xlabel('Overall Score')
ax1.set_ylabel('State')
ax1.set_title('Top 10 States by Quality Score')
ax1.set_xlim(0, 100)

# Bottom 10
bottom_10.plot(kind='barh', x='state', y='overall_score', ax=ax2, color='red', legend=False)
ax2.set_xlabel('Overall Score')
ax2.set_ylabel('State')
ax2.set_title('Bottom 10 States by Quality Score')
ax2.set_xlim(0, 100)

plt.tight_layout()
plt.savefig('../reports/figures/top_bottom_states.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: top_bottom_states.png")

In [ ]:
# Time series of quality scores
time_series = detailed.groupby('period')['overall_score'].mean().sort_index()

fig, ax = plt.subplots(figsize=(12, 6))
time_series.plot(ax=ax, marker='o', linewidth=2, markersize=6)
ax.set_xlabel('Period')
ax.set_ylabel('Average Overall Score')
ax.set_title('Quality Score Trend Over Time')
ax.axhline(time_series.mean(), color='red', linestyle='--', alpha=0.5, 
           label=f"Average: {time_series.mean():.1f}")
ax.legend()
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../reports/figures/quality_trend.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: quality_trend.png")

In [ ]:
# Issue frequency heatmap
flag_cols = [col for col in detailed.columns if col.startswith('flag_')]
issue_rates = detailed[flag_cols].mean() * 100

fig, ax = plt.subplots(figsize=(10, 6))
issue_rates.plot(kind='bar', ax=ax, color='coral')
ax.set_ylabel('Percentage of Records Flagged (%)')
ax.set_xlabel('Issue Type')
ax.set_title('Data Quality Issues - Prevalence Rates')
ax.set_xticklabels([col.replace('flag_', '').replace('_', ' ').title() for col in flag_cols], 
                   rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../reports/figures/issue_prevalence.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: issue_prevalence.png")

## 6. Summary and Recommendations

### Key Findings

1. **Overall Quality**: Average national score is around 75/100 (Grade B), indicating good but improvable quality.

2. **Timeliness is the weakest component**: Nearly 30% of reports are submitted late, with average delays of 5-10 days.

3. **Completeness is strong**: Over 90% of required fields are populated, though 5-7% of records have missing indicators.

4. **Consistency violations**: About 5% of records violate logical relationships (e.g., penta3 > penta1).

5. **State variation**: Top states score 80+, bottom states score 60-70, suggesting uneven capacity.

### Recommendations

**Priority 1 - Improve Timeliness (Low-cost, high-impact)**:
- Implement automated SMS/email reminders at day 25 of month, day 5 after month-end, and day 8 for late facilities
- Create real-time submission dashboard for managers
- Consider 10-day deadline for hard-to-reach facilities

**Priority 2 - Strengthen Validation (Low-cost)**:
- Add built-in validation rules to data entry system (flag penta3 > penta1 before submission)
- Display previous month values during entry for comparison
- Clarify guidance on zero vs. blank entries

**Priority 3 - Targeted Support (Moderate-cost)**:
- Refresher training for bottom 20% facilities on indicator definitions
- Establish peer learning networks
- Quarterly on-site supervision for lowest-scoring LGAs

**Priority 4 - Enhance Feedback (Low-cost, high-impact)**:
- Monthly DQA scorecards to facilities
- Recognize top performers quarterly
- Show facilities how their data is used in decision-making

### Next Steps

1. Run this pipeline on real data (6 months)
2. Validate and adjust thresholds
3. Train state HMIS officers on dashboard
4. Pilot automated reminders in 3 states
5. Integrate validation rules into data entry system

**Target**: Improve national score from 75 to 85+ within 12 months through these interventions.

In [ ]:
print("\n" + "="*70)
print("DQA WALKTHROUGH COMPLETE")
print("="*70)
print("\nOutputs created:")
print("  - Processed data: ../data/processed/")
print("  - Figures: ../reports/figures/")
print("  - Metrics: ../reports/metrics_summary.json")
print("\nNext steps:")
print("  - Launch dashboard: streamlit run ../dashboards/dqa_app.py")
print("  - Review policy memo: ../reports/health_dqa_memo.md")
print("  - Run tests: python -m pytest ../tests/")